In [1]:
# step1_simulate_telematics.py
import numpy as np
import pandas as pd

# reproducibility
rng = np.random.default_rng(42)

# Parameters
n_drivers = 500       # number of drivers
trips_per_driver = 50 # trips per driver

driver_ids = np.arange(1, n_drivers + 1)

rows = []
for driver_id in driver_ids:
    # driver-level latent riskiness
    base_risk = rng.uniform(0.05, 0.30)  # probability of claim in a year
    for trip_id in range(trips_per_driver):
        total_miles = rng.normal(15, 5)   # average ~15 miles per trip
        total_miles = max(total_miles, 1)
        
        avg_speed = rng.normal(40, 10)    # mph
        max_speed = avg_speed + rng.normal(10, 5)
        hard_brakes = rng.poisson(lam=0.2 * total_miles / 10)
        rapid_accels = rng.poisson(lam=0.15 * total_miles / 10)
        
        # Night driving indicator (more risky)
        night_drive = rng.choice([0, 1], p=[0.7, 0.3])
        night_miles = night_drive * rng.uniform(0.2, 0.8) * total_miles
        
        # Simulated label: claim probability influenced by features
        # logistic model with noise
        risk_score = (
            0.02 * (max_speed - 70) +
            0.1 * hard_brakes +
            0.08 * rapid_accels +
            0.05 * night_drive +
            base_risk
        )
        prob_claim = 1 / (1 + np.exp(-risk_score))
        claim = rng.binomial(1, min(max(prob_claim, 0), 1))  # yearly indicator

        rows.append({
            "driver_id": driver_id,
            "trip_id": f"{driver_id}_{trip_id}",
            "total_miles": total_miles,
            "avg_speed": avg_speed,
            "max_speed": max_speed,
            "hard_brakes": hard_brakes,
            "rapid_accels": rapid_accels,
            "night_drive": night_drive,
            "night_miles": night_miles,
            "claim": claim
        })

df = pd.DataFrame(rows)

# Show sample
print(df.head())

# Save to CSV for next steps
df.to_csv("telematics_data.csv", index=False)
print("Saved telematics_data.csv with shape:", df.shape)


   driver_id trip_id  total_miles  avg_speed  max_speed  hard_brakes  \
0          1     1_0     9.800079  47.504512  62.207336            0   
1          1     1_1    19.396990  47.777919  58.108073            1   
2          1     1_2    14.750370  38.151376  44.746729            2   
3          1     1_3    17.154105  61.416476  69.384401            0   
4          1     1_4    10.799218  31.755188  45.008152            0   

   rapid_accels  night_drive  night_miles  claim  
0             1            1     2.713330      1  
1             0            0     0.000000      0  
2             0            0     0.000000      0  
3             0            0     0.000000      1  
4             0            1     4.183833      1  
Saved telematics_data.csv with shape: (25000, 10)


In [2]:
df

,driver_id,trip_id,total_miles,avg_speed,max_speed,hard_brakes,rapid_accels,night_drive,night_miles,claim
0,1,1_0,9.800079,47.504512,62.207336,0,1,1,2.713330,1
1,1,1_1,19.396990,47.777919,58.108073,1,0,0,0.000000,0
2,1,1_2,14.750370,38.151376,44.746729,2,0,0,0.000000,0
3,1,1_3,17.154105,61.416476,69.384401,0,0,0,0.000000,1
4,1,1_4,10.799218,31.755188,45.008152,0,0,1,4.183833,1
...,...,...,...,...,...,...,...,...,...,...
24995,500,500_45,20.778572,46.973233,65.756361,0,0,0,0.000000,0
24996,500,500_46,20.273187,48.031426,56.061023,0,1,0,0.000000,1
24997,500,500_47,21.004695,37.072687,45.343599,1,0,0,0.000000,1
24998,500,500_48,21.440377,37.343661,41.195046,0,1,1,8.732268,1


In [3]:
# step2_aggregate_drivers.py
import pandas as pd

# Load trip-level data
df = pd.read_csv("telematics_data.csv")

# Aggregate by driver_id
agg = df.groupby("driver_id").agg(
    trips=("trip_id", "count"),
    total_miles=("total_miles", "sum"),
    avg_trip_miles=("total_miles", "mean"),
    mean_speed=("avg_speed", "mean"),
    max_speed=("max_speed", "max"),
    hard_brakes_per_mile=("hard_brakes", lambda x: x.sum() / (df.loc[x.index, "total_miles"].sum())),
    rapid_accels_per_mile=("rapid_accels", lambda x: x.sum() / (df.loc[x.index, "total_miles"].sum())),
    night_miles_frac=("night_miles", lambda x: x.sum() / df.loc[x.index, "total_miles"].sum()),
    claim=("claim", "max")  # if any claim occurred for driver, label them as claimant
).reset_index()

# Add derived metrics
agg["miles_per_trip"] = agg["total_miles"] / agg["trips"]

# Save aggregated dataset
agg.to_csv("driver_level_data.csv", index=False)

print("Aggregated dataset shape:", agg.shape)
print(agg.head())



Aggregated dataset shape: (500, 11)
   driver_id  trips  total_miles  avg_trip_miles  mean_speed  max_speed  \
0          1     50   774.037220       15.480744   38.693745  69.384401   
1          2     50   681.557693       13.631154   39.142816  77.092136   
2          3     50   713.194459       14.263889   37.295835  70.355343   
3          4     50   753.209854       15.064197   39.377100  80.839998   
4          5     50   721.842144       14.436843   36.473422  75.876688   

   hard_brakes_per_mile  rapid_accels_per_mile  night_miles_frac  claim  \
0              0.011627               0.011627          0.137191      1   
1              0.011738               0.019074          0.125535      1   
2              0.015424               0.008413          0.129436      1   
3              0.025225               0.011949          0.149830      1   
4              0.011083               0.015239          0.181264      1   

   miles_per_trip  
0       15.480744  
1       13.631154  
2 

In [4]:
agg.head()

,driver_id,trips,total_miles,avg_trip_miles,mean_speed,max_speed,hard_brakes_per_mile,rapid_accels_per_mile,night_miles_frac,claim,miles_per_trip
0,1,50,774.037220,15.480744,38.693745,69.384401,0.011627,0.011627,0.137191,1,15.480744
1,2,50,681.557693,13.631154,39.142816,77.092136,0.011738,0.019074,0.125535,1,13.631154
2,3,50,713.194459,14.263889,37.295835,70.355343,0.015424,0.008413,0.129436,1,14.263889
3,4,50,753.209854,15.064197,39.377100,80.839998,0.025225,0.011949,0.149830,1,15.064197
4,5,50,721.842144,14.436843,36.473422,75.876688,0.011083,0.015239,0.181264,1,14.436843


In [5]:
agg["claim"].value_counts()

claim
1    500
Name: count, dtype: int64

In [9]:
# step3_risk_model.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
# import lightgbm as lgb

# Load driver-level dataset
df = pd.read_csv("driver_level_data.csv")

# Features & target
X = df.drop(columns=["driver_id", "claim"])
y = df["claim"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# --------- Baseline Logistic Regression ---------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train, y)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=200)
log_reg.fit(X_train_scaled, y_train)

y_pred_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]
auc_lr = roc_auc_score(y_test, y_pred_proba_lr)

print("Logistic Regression AUC:", round(auc_lr, 3))
print("Confusion matrix:\n", confusion_matrix(y_test, (y_pred_proba_lr > 0.5).astype(int)))
print("Classification report:\n", classification_report(y_test, (y_pred_proba_lr > 0.5).astype(int)))

# # --------- LightGBM Gradient Boosting ---------
# train_data = lgb.Dataset(X_train, label=y_train)
# test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# params = {
#     "objective": "binary",
#     "metric": "auc",
#     "boosting_type": "gbdt",
#     "num_leaves": 31,
#     "learning_rate": 0.05,
#     "feature_fraction": 0.8,
#     "bagging_fraction": 0.8,
#     "bagging_freq": 5,
#     "verbosity": -1,
# }

# gbm = lgb.train(params, train_data, valid_sets=[test_data], num_boost_round=200, early_stopping_rounds=20)

# y_pred_proba_gbm = gbm.predict(X_test, num_iteration=gbm.best_iteration)
# auc_gbm = roc_auc_score(y_test, y_pred_proba_gbm)

# print("LightGBM AUC:", round(auc_gbm, 3))

# --------- Add risk scores back to dataset ---------
df["risk_score"] = log_reg.predict_proba(scaler.transform(X))[:, 1]

# Save driver dataset with risk scores
df.to_csv("driver_risk_scores.csv", index=False)
print("Saved driver_risk_scores.csv with shape:", df.shape)


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(1)